# 3.4 — グループ化と要約統計

明細行を意思決定に使える表へ集約し、集計単位、件数、分母、欠損、分布を明示して結果を検証します。

## 導入

このNotebookでは、Moodle本文の概念を実際のデータとコードで確かめます。

## このレッスンの到達目標

- 明細と集計結果について、一行が表す粒度を定義できる。
- groupbyと名前付きaggで、件数・合計・統計量・条件付き件数を作れる。
- 率の分子と分母、構成比の合計、順位の比較順を説明できる。
- 明細との照合、CSV保存、再読込により集計結果を検証できる。

> **学習経路:** 必須：3.4.1〜3.4.5　／　統合練習：3.4.6


## 3.4.1 明細と集計結果の粒度を決める

元表の一行はセンター・月・コースの記録です。地区別にまとめるなら結果の一行は一地区、地区・コース別なら一地区・一コースになります。この粒度を決めずに`groupby()`を書くと、同じ数値でも何を数えたか説明できません。

In [ ]:
from pathlib import Path
import pandas as pd

def find_course_data(filename):
    roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "work", Path("/opt/python-lab/course-materials")]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Course data was not found:\n" + "\n".join(map(str, checked)))

raw = pd.read_csv(find_course_data("learning-centres-practice.csv"))
clean = raw.copy()
clean["district"] = clean["district"].astype("string").str.strip().str.title()
business_key = ["centre_id", "month", "course"]
quality_problem = (
    clean["attended"].isna()
    | (clean["completed"] > clean["attended"])
    | clean.duplicated(subset=business_key, keep=False)
)
analysis = clean.loc[~quality_problem].copy()
analysis["completion_rate"] = analysis["completed"] / analysis["registered"] * 100
print("Source:", len(raw), "analysis-ready:", len(analysis), "flagged:", int(quality_problem.sum()))


## 3.4.2 groupbyで分け、数え、まとめる

`groupby(キー)`は同じキーの行を分け、各組へ同じ集計を適用し、結果を結合します。名前付き集計を使うと、出力列の意味をコードに残せます。`reset_index()`はグループキーを通常の列へ戻し、後の表やグラフで扱いやすくします。

In [ ]:
district_summary = (
    analysis.groupby("district", dropna=False)
    .agg(
        centre_months=("centre_id", "size"),
        centres=("centre_id", "nunique"),
        registered_total=("registered", "sum"),
        completed_total=("completed", "sum"),
    )
    .reset_index()
)
district_summary


### size・count・nuniqueの対象を区別する

`size`は欠損を含む行数、`count`は指定列の欠損でない値の数、`nunique`は異なる値の数です。「センター月数」「報告済み出席者数」「センター数」は別の指標なので、質問に合うものを選びます。

In [ ]:
count_check = analysis.groupby("course").agg(
    rows=("centre_id", "size"),
    reported_attendance=("attended", "count"),
    distinct_centres=("centre_id", "nunique"),
)
count_check


### 条件付き件数を名前付き集計へ入れる

「修了率75%未満だった記録数」は単なる行数ではありません。まず各明細が条件に当てはまるかをブール列にし、グループ内でTrueを合計します。判定と集計を分けると、条件そのものと件数を別々に確認できます。


In [ ]:
operational = analysis.assign(low_completion=analysis["completion_rate"] < 75)
condition_summary = operational.groupby("course", as_index=False).agg(
    records=("centre_id", "size"),
    low_completion_records=("low_completion", "sum"),
)
condition_summary


## 3.4.3 合計・統計量・率を計算する

`sum`は総量、`mean`は算術平均、`median`は並べた中央、`min`と`max`は範囲の端、`std`は平均からの散らばりを表します。平均は極端な値の影響を受けやすいため、件数、中央値、最小・最大と一緒に読みます。標準偏差は単位を保ちますが、原因までは説明しません。

In [ ]:
distribution = analysis.groupby("course")["registered"].agg(
    ["size", "sum", "mean", "median", "min", "max", "std"]
)
distribution.round(2)


### 対応する分子と分母の合計から率を求める

コース全体の修了率は、修了者合計を登録者合計で割ります。各センター月の率を単純平均すると、小規模行と大規模行へ同じ重みを与えるため、全参加者の率とは一致しません。どちらが正しいかではなく、問いが「典型的なセンター月」か「全登録者」かで決まります。

In [ ]:
course_summary = analysis.groupby("course").agg(
    records=("centre_id", "size"),
    registered_total=("registered", "sum"),
    completed_total=("completed", "sum"),
    mean_row_completion_rate=("completion_rate", "mean"),
    median_row_completion_rate=("completion_rate", "median"),
    material_cost_total=("material_cost", "sum"),
)
course_summary["overall_completion_rate"] = (
    course_summary["completed_total"] / course_summary["registered_total"] * 100
)
course_summary["cost_per_completion"] = (
    course_summary["material_cost_total"] / course_summary["completed_total"]
)
course_summary.round(2)


### 複数キーで比較の階層を保つ

地区だけの集計と地区・コースの集計は粒度が違います。複数キーでまとめ、`sort_values()`で順序を明示します。結果を結合するときは、粒度の違う表を無条件に足したり平均したりしません。

In [ ]:
district_course = (
    analysis.groupby(["district", "course"], dropna=False)
    .agg(records=("centre_id", "size"), registered=("registered", "sum"), completed=("completed", "sum"))
    .reset_index()
)
district_course["completion_rate"] = district_course["completed"] / district_course["registered"] * 100
district_course.sort_values(["district", "course"]).round(2)


## 3.4.4 判断に使う指標と順位を作る

優先順位には複数の規則が必要です。ここでは一人修了当たり教材費が高い順、同じなら記録数が多い順、さらに同じならコース名順とします。最後の安定した識別列まで指定すると、同じデータから毎回同じ先頭行を得られます。丸める値がある場合は、丸める前の値で順位を決めます。


In [ ]:
ranked_course = (
    course_summary.reset_index()
    .sort_values(
        ["cost_per_completion", "records", "course"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
ranked_course.insert(0, "priority", range(1, len(ranked_course) + 1))
ranked_course[["priority", "course", "cost_per_completion", "records"]].round(2)


### 構成比の分母と100%を確認する

地区内の登録者構成比なら、各地区・コースの登録者数をその地区の登録者合計で割ります。全体合計を分母にすれば別の問いになります。`transform('sum')`は元の各行へ所属グループの合計を対応させ、地区ごとの構成比合計を検証できます。

In [ ]:
district_course["district_registered_total"] = district_course.groupby("district")["registered"].transform("sum")
district_course["share_within_district"] = district_course["registered"] / district_course["district_registered_total"] * 100
print(district_course.groupby("district")["share_within_district"].sum().round(6))


## 3.4.5 集計結果を照合し、保存後に再確認する

グループ別合計をもう一度合計すると、分析対象全体の合計と一致するはずです。行数、登録者数、修了者数、教材費を照合します。小さなグループの平均には件数を添え、差が原因や優劣を証明するとは解釈しません。

In [ ]:
assert int(course_summary["records"].sum()) == len(analysis)
assert course_summary["registered_total"].sum() == analysis["registered"].sum()
assert course_summary["completed_total"].sum() == analysis["completed"].sum()
assert abs(course_summary["material_cost_total"].sum() - analysis["material_cost"].sum()) < 1e-9
print("Reconciliation passed")


### 用途別CSVを保存し、再読込して検証する

提出物になるCSVは、Notebook内のDataFrameとは別の境界です。二つの用途別CSVを保存し、再読込後の列、件数、先頭の優先対象を照合します。直前の変数が正しくても、保存列や並びを誤れば成果物は正しくありません。


In [ ]:
review_columns = ["month", "centre_id", "course", "registered", "completed", "completion_rate"]
review_output = (
    operational.loc[operational["low_completion"], review_columns]
    .sort_values(["month", "centre_id", "course"])
    .reset_index(drop=True)
)
summary_output = ranked_course.copy()

output_dir = Path.cwd() / "output" / "lesson34"
output_dir.mkdir(parents=True, exist_ok=True)
review_path = output_dir / "records_to_review.csv"
summary_path = output_dir / "course_priority_summary.csv"
review_output.to_csv(review_path, index=False)
summary_output.to_csv(summary_path, index=False)

saved_review = pd.read_csv(review_path)
saved_summary = pd.read_csv(summary_path)
assert list(saved_review.columns) == review_columns
assert len(saved_review) == len(review_output)
assert list(saved_summary.columns) == list(summary_output.columns)
assert len(saved_summary) == len(summary_output)
assert saved_summary.iloc[0]["course"] == summary_output.iloc[0]["course"]
print("Saved-output reconciliation passed:", review_path, summary_path)


## 3.4.6 統合練習：別の判断に必要な集計を作る

月別・コース別に、記録数、異なるセンター数、登録者合計、出席者合計、修了者合計、全体修了率、教材費合計、一人修了当たり教材費を求めてください。各率の分母を文章で書き、単純な行別率平均とも比較し、全体合計との照合を追加します。件数が小さい比較を一つ指摘してください。

In [ ]:
# ここに応用練習の解答を書きます。


## まとめ

- 先に結果表の粒度を決めてからgroupbyを行いました。
- 件数、統計量、率、構成比、順位を、それぞれの定義と分母に結び付けました。
- 明細との合計照合と再読込により、保存された成果物まで検証しました。

## 3.5A 中間実践課題へ

3.5Aでは、原資料を確認する小プログラムと、品質判定・集計・順位付けを行う本番プログラムを完成させます。3.1〜3.4の処理を初めて一つの意思決定へ接続します。

**学習時間の目安:** 約3時間
